# Lab 5: Word Embeddings and Sentence Similarity

Tasks:
1. Train **Word2Vec**, **FastText**, and **Doc2Vec** on the training data (from Lab 4 / 1 M sentence corpus).
2. **K-Means clustering** of word vectors — represent each cluster with 20 words closest to the centroid.
3. Represent each sentence in the **validation and test sets** as a vector (word-embedding average + Doc2Vec) and find the **closest sentence** in the training set for each.

In [1]:
# Cell 1 – Imports & reproduce train/dev/test split from Lab 3/4
import os, re, random
from collections import Counter

import numpy as np
from gensim.models import Word2Vec, FastText
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.cluster import KMeans

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CORPUS_FILE = 'hindi_1_1m_sentences.txt'

# ── Reproduce the same split used in Lab 4 ──────────────────────────────────
with open(CORPUS_FILE, encoding='utf-8') as f:
    all_sentences = [l.strip() for l in f if l.strip()]
print(f'Total sentences loaded: {len(all_sentences):,}')

def length_balanced_split(sentences, dev_size=1000, test_size=1000, seed=SEED):
    rng = random.Random(seed)
    sorted_sents = sorted(sentences, key=lambda s: len(s.split()))
    total  = len(sorted_sents)
    decile = total // 10
    dper   = dev_size  // 10
    tper   = test_size // 10
    dev_set, test_set, train_set = [], [], []
    for d in range(10):
        sl = sorted_sents[d*decile : (d+1)*decile if d < 9 else total]
        rng.shuffle(sl)
        dev_set.extend(sl[:dper])
        test_set.extend(sl[dper : dper + tper])
        train_set.extend(sl[dper + tper:])
    return train_set, dev_set, test_set

train_sents, dev_sents, test_sents = length_balanced_split(all_sentences)
print(f'Train: {len(train_sents):,} | Dev: {len(dev_sents):,} | Test: {len(test_sents):,}')

# Tokenise: simple whitespace split (same as Lab 4)
def tokenise(sent):
    return sent.strip().split()

train_tokens = [tokenise(s) for s in train_sents]
dev_tokens   = [tokenise(s) for s in dev_sents]
test_tokens  = [tokenise(s) for s in test_sents]

print(f'Tokenisation done. Example: {train_tokens[0][:8]}')

Total sentences loaded: 1,100,000
Train: 1,098,000 | Dev: 1,000 | Test: 1,000
Tokenisation done. Example: ['जबकि', 'अस्पताल', 'में', 'मरीज', 'भर्ती', 'थे।']


## 1a. Train Word2Vec

Reference: https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html

In [2]:
# Cell 2 – Train Word2Vec
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=5,
    seed=SEED
)

print(f'Word2Vec vocabulary size: {len(w2v_model.wv):,}')
print(f'Vector size: {w2v_model.vector_size}')

# Quick sanity check – most similar words to a common Hindi word
sample_word = list(w2v_model.wv.key_to_index.keys())[100]
print(f'\nTop-5 words most similar to "{sample_word}":')
for word, score in w2v_model.wv.most_similar(sample_word, topn=5):
    print(f'  {word}: {score:.4f}')

Word2Vec vocabulary size: 83,775
Vector size: 100

Top-5 words most similar to "कोरोना":
  कोविड-19: 0.8622
  कोविड: 0.8070
  कोरोनावायरस: 0.8001
  कोराना: 0.7778
  संक्रमण: 0.7243


## 1b. Train FastText

Reference: https://radimrehurek.com/gensim/auto_examples/tutorials/run_fasttext.html

In [3]:
# Cell 3 – Train FastText
ft_model = FastText(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=5,
    seed=SEED
)

print(f'FastText vocabulary size: {len(ft_model.wv):,}')
print(f'Vector size: {ft_model.vector_size}')

# Sanity check
sample_word = list(ft_model.wv.key_to_index.keys())[100]
print(f'\nTop-5 words most similar to "{sample_word}":')
for word, score in ft_model.wv.most_similar(sample_word, topn=5):
    print(f'  {word}: {score:.4f}')

FastText vocabulary size: 83,775
Vector size: 100

Top-5 words most similar to "कोरोना":
  “कोरोना: 0.9858
  -कोरोना: 0.9850
  ‘कोरोना: 0.9846
  (कोरोना: 0.9843
  'कोरोना: 0.9820


## 1c. Train Doc2Vec

Reference: https://radimrehurek.com/gensim/auto_examples/tutorials/run_doc2vec_lee.html

In [4]:
# Cell 4 – Train Doc2Vec
tagged_train = [
    TaggedDocument(words=words, tags=[i])
    for i, words in enumerate(train_tokens)
]

d2v_model = Doc2Vec(
    documents=tagged_train,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=20,
    seed=SEED
)

print(f'Doc2Vec vocabulary size: {len(d2v_model.wv):,}')
print(f'Vector size: {d2v_model.vector_size}')
print(f'Number of document vectors: {len(d2v_model.dv):,}')

Doc2Vec vocabulary size: 83,775
Vector size: 100
Number of document vectors: 1,098,000


## 2a. K-Means Clustering of Word Vectors

Cluster the Word2Vec word vectors using K-Means. Represent each cluster with the 20 words closest to its centroid.

In [5]:
# Cell 5 – K-Means clustering on Word2Vec vectors
K = 100  # number of clusters

# Collect all word vectors
words_list  = list(w2v_model.wv.key_to_index.keys())
word_matrix = w2v_model.wv.vectors  # shape: (vocab, vector_size)

print(f'Running K-Means with K={K} on {word_matrix.shape[0]:,} word vectors...')
kmeans = KMeans(n_clusters=K, random_state=SEED, n_init=10)
kmeans.fit(word_matrix)

labels    = kmeans.labels_           # cluster assignment per word
centroids = kmeans.cluster_centers_  # shape: (K, vector_size)

print('Clustering done.')

# ── For each cluster: find 20 words closest to the centroid ─────────────────
from sklearn.metrics.pairwise import cosine_similarity

cluster_representatives = {}
for c in range(K):
    # Indices of words assigned to this cluster
    member_indices = np.where(labels == c)[0]
    member_vecs    = word_matrix[member_indices]
    centroid       = centroids[c].reshape(1, -1)

    # Cosine similarity of each member to the centroid
    sims = cosine_similarity(centroid, member_vecs)[0]

    # Top-20 (or fewer if cluster is small)
    top_n = min(20, len(member_indices))
    top_idx = np.argsort(sims)[::-1][:top_n]
    top_words = [words_list[member_indices[i]] for i in top_idx]
    cluster_representatives[c] = top_words

# Display first 5 clusters
print('\n--- Sample Clusters (first 5) ---')
for c in range(5):
    print(f'\nCluster {c}: {cluster_representatives[c]}')

Running K-Means with K=100 on 83,775 word vectors...
Clustering done.

--- Sample Clusters (first 5) ---

Cluster 0: ['निकालते', 'जमे', 'जोड़ते', 'रोकते', 'बचाते', 'बजाते', 'सोए', 'काटते', 'खींचते', 'खोलते', 'डरे', 'गुजरते', 'छोड़ते', 'बंधे', 'हटाते', 'पढ़ते', 'लड़ते', 'लाते', 'जलते', 'भरते']

Cluster 1: ['धनबाद।', 'मुम्बई।', 'सोनीपत।', 'आइएएनएस।', 'बेंगलुरू।', 'सीकर।', 'AFP)', 'गांधीनगर।', 'भिवानी।', 'रॉयटर्स)', 'News18)', 'बहादुरगढ़।', 'मेरठ।', 'एजेंसियां।', 'चंडीगढ़।', 'कटनी।', 'न्यूयॉर्क।', 'जालंधर।', 'रायटर।', 'बलरामपुर।']

Cluster 2: ['सॉन्ग', 'कॉमेडी', 'स्टाइल', 'लुक', 'गाने', 'सीन', 'गानों', 'फिल्\u200dम', 'रोमांटिक', 'हॉलीवुड', 'फैन', 'एक्टिंग', 'डांस', 'शो', 'अभिनय', 'फेमस', 'फ़िल्म', 'फोटोशूट', 'बोल्ड', 'बॉलीवुड']

Cluster 3: ['लोगों', 'कर्मचारियों', 'श्रमिकों', 'मजदूरों', 'छात्रों', 'लोगो', 'व्यक्तियों', 'नागरिकों', 'कर्मियों', 'परिवारों', 'बच्चों', 'विद्यार्थियों', 'युवाओं', 'ग्रामीणों', 'शिक्षकों', 'छात्र-छात्राओं', 'महिलाओं', 'किसानों', 'दुकानदारों', 'यात्रियों']

Cluster 

In [6]:
# Cell 6 – Print all clusters
print('=== All K-Means Clusters (20 representative words each) ===')
for c in range(K):
    words_str = ', '.join(cluster_representatives[c])
    print(f'Cluster {c:3d}: {words_str}')

=== All K-Means Clusters (20 representative words each) ===
Cluster   0: निकालते, जमे, जोड़ते, रोकते, बचाते, बजाते, सोए, काटते, खींचते, खोलते, डरे, गुजरते, छोड़ते, बंधे, हटाते, पढ़ते, लड़ते, लाते, जलते, भरते
Cluster   1: धनबाद।, मुम्बई।, सोनीपत।, आइएएनएस।, बेंगलुरू।, सीकर।, AFP), गांधीनगर।, भिवानी।, रॉयटर्स), News18), बहादुरगढ़।, मेरठ।, एजेंसियां।, चंडीगढ़।, कटनी।, न्यूयॉर्क।, जालंधर।, रायटर।, बलरामपुर।
Cluster   2: सॉन्ग, कॉमेडी, स्टाइल, लुक, गाने, सीन, गानों, फिल्‍म, रोमांटिक, हॉलीवुड, फैन, एक्टिंग, डांस, शो, अभिनय, फेमस, फ़िल्म, फोटोशूट, बोल्ड, बॉलीवुड
Cluster   3: लोगों, कर्मचारियों, श्रमिकों, मजदूरों, छात्रों, लोगो, व्यक्तियों, नागरिकों, कर्मियों, परिवारों, बच्चों, विद्यार्थियों, युवाओं, ग्रामीणों, शिक्षकों, छात्र-छात्राओं, महिलाओं, किसानों, दुकानदारों, यात्रियों
Cluster   4: मर, रुक, बह, फंस, थम, निकल, उड़, आ, छा, टूट, फैल, गुजर, बढ़, बच, बिगड़, जुड़, चढ़, डूब, गिर, खुल
Cluster   5: ईरान, अमरीका, रूस, अमेरिका, अफगानिस्तान, चीन, तुर्की, सीरिया, पाकिस्तान, तालिबान, फ्रांस, जापान, जर्म

## 2b. Sentence Representation & Nearest Sentence Retrieval

For each sentence in the **dev** and **test** sets:
- Represent it as a vector using:
  - **Word2Vec average** (mean of word vectors)
  - **FastText average** (mean of word vectors)
  - **Doc2Vec** (inferred document vector)
- Find the **closest sentence** in the training set (cosine similarity).

In [7]:
# Cell 7 – Build training sentence vectors for retrieval
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

DIM = 100

def avg_word_vec(tokens, wv_model):
    """Mean of word vectors for words present in the model vocabulary."""
    vecs = [wv_model.wv[w] for w in tokens if w in wv_model.wv]
    if not vecs:
        return np.zeros(DIM)
    return np.mean(vecs, axis=0)

def doc2vec_vec(tokens, model):
    """Infer a vector for an unseen document."""
    return model.infer_vector(tokens)

print('Building training set sentence matrices ...')

train_w2v_matrix = np.array([avg_word_vec(t, w2v_model) for t in train_tokens])  # (N_train, 100)
train_ft_matrix  = np.array([avg_word_vec(t, ft_model)  for t in train_tokens])  # (N_train, 100)
train_d2v_matrix = np.array([d2v_model.dv[i] for i in range(len(train_tokens))]) # (N_train, 100)

print(f'Train matrices: W2V {train_w2v_matrix.shape}, FT {train_ft_matrix.shape}, D2V {train_d2v_matrix.shape}')

Building training set sentence matrices ...
Train matrices: W2V (1098000, 100), FT (1098000, 100), D2V (1098000, 100)


In [8]:
# Cell 8 – Nearest-sentence retrieval function
def find_closest_sentence(query_vec, train_matrix, train_sentences, topn=1):
    """Return the topn closest training sentences by cosine similarity."""
    sims = cos_sim(query_vec.reshape(1, -1), train_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:topn]
    return [(train_sentences[i], sims[i]) for i in top_idx]

def retrieve_closest(eval_tokens, eval_sentences, train_matrix, train_sentences, embed_fn, label):
    """For each sentence in eval set, find and print the closest training sentence."""
    print(f'\n=== Closest sentence using {label} ===')
    for i, (tokens, query_sent) in enumerate(zip(eval_tokens, eval_sentences)):
        q_vec = embed_fn(tokens)
        closest = find_closest_sentence(q_vec, train_matrix, train_sentences, topn=1)
        closest_sent, score = closest[0]
        print(f'\n[{i+1}] Query  : {query_sent}')
        print(f'     Closest: {closest_sent}')
        print(f'     Sim    : {score:.4f}')

In [9]:
# Cell 9 – Closest sentence for DEV set using Word2Vec average
print('--- DEV SET ---')
retrieve_closest(
    dev_tokens, dev_sents,
    train_w2v_matrix, train_sents,
    lambda t: avg_word_vec(t, w2v_model),
    label='Word2Vec (avg)'
)

--- DEV SET ---

=== Closest sentence using Word2Vec (avg) ===

[1] Query  : एडमिशन सेल
     Closest: एडमिशन कोऑर्डिनेटर प्रो।
     Sim    : 0.8042

[2] Query  : उन्होंने कहा, 'निश्चित रूप से।
     Closest: उन्होंने कहा, "सरकार देश...
     Sim    : 0.8184

[3] Query  : अनंता, 26.
     Closest: 26.
     Sim    : 1.0000

[4] Query  : दर्शनार्थियों के लिए भी रुट है डिसाइड
     Closest: और मल्टी टास्किंग के लिए मौज़ूद है 4GB RAM।
     Sim    : 0.9339

[5] Query  : हाईस्कूल का परिणाम 96 फीसदी रहा।
     Closest: बारहवीं का परिणाम 13 प्रतिशत कम रहा।
     Sim    : 0.9300

[6] Query  : भावुकता में निर्णय लेने से बचें।
     Closest: जल्दबाजी में निर्णय लेने से बचें।
     Sim    : 0.9919

[7] Query  : आरयू एकेडमिक काउंसिल की बैठक आज
     Closest: आज गरबा मंडलों की बैठक
     Sim    : 0.9193

[8] Query  : धारावाहिक में वह दोनों भाई-बहन थे।
     Closest: वे दोनों नागपुर में ही थे।
     Sim    : 0.9393

[9] Query  : पूर्व आवासीय मंत्री के।
     Closest: पूर्व मंत्री स्व.
     Sim    : 0.9670

[10] Qu

In [10]:
# Cell 10 – Closest sentence for DEV set using FastText average
print('--- DEV SET ---')
retrieve_closest(
    dev_tokens, dev_sents,
    train_ft_matrix, train_sents,
    lambda t: avg_word_vec(t, ft_model),
    label='FastText (avg)'
)

--- DEV SET ---

=== Closest sentence using FastText (avg) ===

[1] Query  : एडमिशन सेल
     Closest: सर्टिफाइड इंर्फोमेशन सिस्टम सिक्योरिटी प्रोफेशन
     Sim    : 0.8537

[2] Query  : उन्होंने कहा, 'निश्चित रूप से।
     Closest: * देखहिं रूप महा रनधीरा।
     Sim    : 0.8372

[3] Query  : अनंता, 26.
     Closest: अनंता, 103.
     Sim    : 0.9414

[4] Query  : दर्शनार्थियों के लिए भी रुट है डिसाइड
     Closest: बच्चों के लिए भी है उपयोगी
     Sim    : 0.9844

[5] Query  : हाईस्कूल का परिणाम 96 फीसदी रहा।
     Closest: ग्वालियर का परीक्षा परिणाम 63.23 प्रतिशत रहा।
     Sim    : 0.9584

[6] Query  : भावुकता में निर्णय लेने से बचें।
     Closest: जल्दबाजी में निर्णय लेने से बचें।
     Sim    : 0.9905

[7] Query  : आरयू एकेडमिक काउंसिल की बैठक आज
     Closest: आज टास्क फोर्स की बैठक
     Sim    : 0.9604

[8] Query  : धारावाहिक में वह दोनों भाई-बहन थे।
     Closest: वह आनलाइन क्लास में थे।
     Sim    : 0.9440

[9] Query  : पूर्व आवासीय मंत्री के।
     Closest: पूर्व केंद्रीय मंत्री स्व.
   

In [11]:
# Cell 11 – Closest sentence for DEV set using Doc2Vec
print('--- DEV SET ---')
retrieve_closest(
    dev_tokens, dev_sents,
    train_d2v_matrix, train_sents,
    lambda t: doc2vec_vec(t, d2v_model),
    label='Doc2Vec (inferred)'
)

--- DEV SET ---

=== Closest sentence using Doc2Vec (inferred) ===

[1] Query  : एडमिशन सेल
     Closest: काउंसलिंग सेल के कोओर्डिनेटर प्रो.
     Sim    : 0.5551

[2] Query  : उन्होंने कहा, 'निश्चित रूप से।
     Closest: के रूप में अच्छी तरह से।
     Sim    : 0.8108

[3] Query  : अनंता, 26.
     Closest: जागरण संवाददाता, चंडीगढ़।
     Sim    : 0.8608

[4] Query  : दर्शनार्थियों के लिए भी रुट है डिसाइड
     Closest: रात्रि १.४५ तक है, जो शुभकार्यारम्भ के लिए मार्ग प्रशस्त करता है।
     Sim    : 0.6131

[5] Query  : हाईस्कूल का परिणाम 96 फीसदी रहा।
     Closest: परिणाम सफल रहेंगे।
     Sim    : 0.7429

[6] Query  : भावुकता में निर्णय लेने से बचें।
     Closest: जागरण संवाददाता, चंडीगढ़।
     Sim    : 0.8535

[7] Query  : आरयू एकेडमिक काउंसिल की बैठक आज
     Closest: जागरण संवाददाता, चंडीगढ़।
     Sim    : 0.8077

[8] Query  : धारावाहिक में वह दोनों भाई-बहन थे।
     Closest: वह 79 साल के थे.
     Sim    : 0.8043

[9] Query  : पूर्व आवासीय मंत्री के।
     Closest: आगे बढ़कर तन-तन के।
     S

In [12]:
# Cell 12 – Closest sentence for TEST set using Word2Vec average
print('--- TEST SET ---')
retrieve_closest(
    test_tokens, test_sents,
    train_w2v_matrix, train_sents,
    lambda t: avg_word_vec(t, w2v_model),
    label='Word2Vec (avg)'
)

--- TEST SET ---

=== Closest sentence using Word2Vec (avg) ===

[1] Query  : वह ईश्वर,
     Closest: वह जी.डी.पी.
     Sim    : 0.9979

[2] Query  : मेड्रिड।
     Closest: खदान मालिकों द्वारा दस्तावेज पेश करने में की जा रही देरी की पुष्टि करते हुए गोवा मिनरल ओर एक्सपोर्टर्स एसोसिएशन (जीएमओईए) के सचिव ग्लेन कलावमपारा ने कहा, 'मॉनसून के दौरान खदान का काम सुस्त पड़ने के कारण खदान मालिक किसी जल्दबाजी में नहीं हैं।'
     Sim    : 0.0000

[3] Query  : यात्रा सफल रहेगी।
     Closest: यात्रा सफल रहेगी।
     Sim    : 1.0000

[4] Query  : संतकबीरनगर।
     Closest: संतकबीरनगर।
     Sim    : 1.0000

[5] Query  : दिसम्‍बर 1930 में ताजमहल पर मंडराया था
     Closest: घाटे में था लैंडमार्क
     Sim    : 0.8810

[6] Query  : जाम भी लगता है।
     Closest: इससे जाम लगता है।
     Sim    : 0.8959

[7] Query  : अभी ऎसा नहीं हो रहा है।
     Closest: किंतु, यह नहीं हो रहा है।
     Sim    : 0.9296

[8] Query  : पीपलझोपा में लगा भौंगर्या हाट
     Closest: हुलास नगरा में लगा जाम।
     Sim    : 0.9807

[9] Query 

In [13]:
# Cell 13 – Closest sentence for TEST set using FastText average
print('--- TEST SET ---')
retrieve_closest(
    test_tokens, test_sents,
    train_ft_matrix, train_sents,
    lambda t: avg_word_vec(t, ft_model),
    label='FastText (avg)'
)

--- TEST SET ---

=== Closest sentence using FastText (avg) ===

[1] Query  : वह ईश्वर,
     Closest: वह जी.डी.पी.
     Sim    : 0.9758

[2] Query  : मेड्रिड।
     Closest: मेड्रिड।
     Sim    : 1.0000

[3] Query  : यात्रा सफल रहेगी।
     Closest: यात्रा सफल रहेगी।
     Sim    : 1.0000

[4] Query  : संतकबीरनगर।
     Closest: संतकबीरनगर।
     Sim    : 1.0000

[5] Query  : दिसम्‍बर 1930 में ताजमहल पर मंडराया था
     Closest: जनवरी में नीतीश पर फेंका था जूता
     Sim    : 0.9326

[6] Query  : जाम भी लगता है।
     Closest: इससे जाम लगता है।
     Sim    : 0.9224

[7] Query  : अभी ऎसा नहीं हो रहा है।
     Closest: अब वहाँ जनजीवन सामान्य हो रहा है.
     Sim    : 0.9517

[8] Query  : पीपलझोपा में लगा भौंगर्या हाट
     Closest: मंझलाचुंबा में लगा जतरा मेला
     Sim    : 0.9452

[9] Query  : अजमेर।
     Closest: अजमेर।
     Sim    : 1.0000

[10] Query  : विदेशी मुद्रा व्यापार.
     Closest: विदेशी मुद्रा युक्तियाँ.
     Sim    : 0.9726

[11] Query  : अदालत ने विशेष जांच दल...
     Closest: एसडी

In [14]:
# Cell 14 – Closest sentence for TEST set using Doc2Vec
print('--- TEST SET ---')
retrieve_closest(
    test_tokens, test_sents,
    train_d2v_matrix, train_sents,
    lambda t: doc2vec_vec(t, d2v_model),
    label='Doc2Vec (inferred)'
)

--- TEST SET ---

=== Closest sentence using Doc2Vec (inferred) ===

[1] Query  : वह ईश्वर,
     Closest: 91 के कुल स्कोर पर वह आउट हो गए.
     Sim    : 0.7829

[2] Query  : मेड्रिड।
     Closest: खाटूश्यामजी।
     Sim    : 0.3938

[3] Query  : यात्रा सफल रहेगी।
     Closest: जागरण संवाददाता, चंडीगढ़।
     Sim    : 0.8816

[4] Query  : संतकबीरनगर।
     Closest: जागरण संवाददाता, चंडीगढ़।
     Sim    : 0.8834

[5] Query  : दिसम्‍बर 1930 में ताजमहल पर मंडराया था
     Closest: क्या था साध्वी प्रज्ञा का बयान?
     Sim    : 0.6293

[6] Query  : जाम भी लगता है।
     Closest: जागरण संवाददाता, चंडीगढ़।
     Sim    : 0.8203

[7] Query  : अभी ऎसा नहीं हो रहा है।
     Closest: बाकी लोग अभी भी इससे बचने का प्रयास कर रहे हैं।
     Sim    : 0.7275

[8] Query  : पीपलझोपा में लगा भौंगर्या हाट
     Closest: दक्षिण अफ्रीका पर चोकर्स का ठप्पा लगा हुआ है।
     Sim    : 0.7849

[9] Query  : अजमेर।
     Closest: जागरण संवाददाता, चंडीगढ़।
     Sim    : 0.8665

[10] Query  : विदेशी मुद्रा व्यापार.
     Closest